# 06 · Tablas de resultados para la memoria

Proyecto de titulación · **Maestría en Inteligencia Artificial Aplicada (UDLA)**
Amapola Technologies Corp.

Consolida los resultados dispersos en los notebooks 03 y 04 en **tablas exportables**, listas para
pegar en el documento. Cada tabla se guarda en `outputs/tablas/` como CSV.

Este notebook **recalcula** los resultados en lugar de copiarlos: así las tablas del documento y el
código que las produce no pueden desincronizarse.

| Tabla | Contenido | Responde a |
|---|---|---|
| 1 | Ablación por bloques de variables | ¿Cuánto aporta lo conversacional? |
| 2 | Comparativa de métricas por configuración | ¿Qué modelo rinde mejor? |
| 3 | Significancia estadística de las diferencias | ¿Las diferencias son reales? |
| 4 | Composición de los bloques de variables | ¿Qué contiene cada bloque? |

### Nota de reproducibilidad

Las cifras de XGBoost pueden diferir en el tercer o cuarto decimal respecto de las de los notebooks
03 y 04. **No es un error**: `tree_method="hist"` acumula en punto flotante en el orden en que
terminan los hilos, así que un mismo modelo con la misma semilla da resultados ligeramente distintos
según el número de hilos (`n_jobs`). Las de la regresión logística sí son idénticas, porque es
determinista.

Este notebook fija `n_jobs=4` en todos los ajustes, de modo que **sus tablas son internamente
consistentes entre sí**. Son estas las que deben citarse en el documento: proceden todas de la misma
ejecución y del mismo conjunto de condiciones. Si se necesita reproducibilidad bit a bit, hay que
fijar `n_jobs` y declararlo junto con la versión de XGBoost.

## 0 · Configuración

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import xgboost as xgb

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 240)
SEMILLA = 42

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_OUTPUTS = RAIZ / "outputs"
DIR_TABLAS = DIR_OUTPUTS / "tablas"
DIR_TABLAS.mkdir(parents=True, exist_ok=True)

def exportar(tabla, nombre, indice=False):
    ruta = DIR_TABLAS / f"{nombre}.csv"
    tabla.to_csv(ruta, index=indice, encoding="utf-8-sig")  # BOM: Excel abre bien los acentos
    print(f"tabla -> {ruta.relative_to(RAIZ)}")
    return tabla

In [2]:
paquete = joblib.load(RAIZ / "modelos" / "xgboost_cumplimiento_v1.pkl")
SPW = float(paquete["scale_pos_weight"])
MES_CORTE = paquete["mes_corte"]

# Cada conjunto de variables tiene SUS hiperparametros, buscados por separado en el
# notebook 03. Reutilizar los del operativo para el completo perjudicaria a XGBoost
# de forma artificial y falsearia la comparacion.
metricas_03 = json.loads((DIR_OUTPUTS / "metricas_modelo_v1.json").read_text(encoding="utf-8"))
HIPER_OPERATIVO = metricas_03["hiperparametros"]["operativo · CV temporal"]
HIPER_COMPLETO = metricas_03["hiperparametros"]["completo · CV temporal"]
print("Hiperparametros operativo:", HIPER_OPERATIVO)
print("Hiperparametros completo :", HIPER_COMPLETO)

df = pd.read_csv(DIR_OUTPUTS / "dataset_modelado_v1.csv",
                 dtype={"mes": str, "id_anonimo": str})
TARGET = "cumplimiento_conciliado"
CATEGORICAS = ["ultima_intencion_antes_compromiso", "tipo_operacion", "bucket"]

tr, te = df[df["mes"] < MES_CORTE], df[df["mes"] >= MES_CORTE]
y_tr, y_te = tr[TARGET].to_numpy(), te[TARGET].to_numpy()
TASA_BASE = y_te.mean()

print(f"Dataset      : {len(df):,} cliente-mes | {df['id_anonimo'].nunique():,} clientes")
print(f"Train        : {len(tr):,} filas ({tr['mes'].min()}-{tr['mes'].max()}) | positivos {y_tr.mean():.2%}")
print(f"Test         : {len(te):,} filas ({te['mes'].min()}-{te['mes'].max()}) | positivos {y_te.mean():.2%}")
print(f"scale_pos_weight: {SPW:.3f}")

Hiperparametros operativo: {'subsample': 0.85, 'reg_lambda': 1, 'n_estimators': 600, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.02, 'gamma': 0, 'colsample_bytree': 1.0}
Hiperparametros completo : {'subsample': 0.85, 'reg_lambda': 20, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 0.6}
Dataset      : 18,580 cliente-mes | 9,906 clientes
Train        : 11,078 filas (202507-202601) | positivos 22.68%
Test         : 7,502 filas (202602-202604) | positivos 17.25%
scale_pos_weight: 3.408


---
## 1 · Composición de los bloques de variables

Antes de la ablación conviene fijar qué contiene cada bloque, porque la tabla de resultados no se
entiende sin esto.

In [3]:
BLOQUES = {
    "cartera": {
        "dias_mora_inicial": "Días de mora al inicio del mes",
        "meses_consecutivos_mora": "Meses calendario consecutivos en mora",
        "mes_campania": "Identificador de campaña del mes",
        "tipo_operacion": "TC (tarjeta) o CARTERA",
        "bucket": "Tramo de mora asignado por el banco (B1–B7)",
    },
    "conversacional": {
        "total_intenciones": "Intenciones activadas en el mes",
        "intenciones_distintas": "Intenciones únicas en el mes",
        "activo_compromiso": "Activó intención de compromiso de pago",
        "dias_compromiso": "Plazo elegido (1, 3 o 5 días)",
        "ultima_intencion_antes_compromiso": "Intención previa al primer compromiso",
        "hora_primera_interaccion": "Hora de la primera interacción (0-23)",
        "dia_semana_primera_interaccion": "Día de la semana (0=lunes)",
        "tiene_conversacion": "El cliente-mes tuvo registro en el chatbot",
    },
}

filas = [{"bloque": b, "variable": v, "descripcion": d, "origen":
          "BigQuery" if b == "cartera" else "MySQL (chatbot)"}
         for b, vs in BLOQUES.items() for v, d in vs.items()]
tabla4 = pd.DataFrame(filas)
display(tabla4)
exportar(tabla4, "tabla4_composicion_bloques")

CARTERA = list(BLOQUES["cartera"])
CONVERSACIONAL = list(BLOQUES["conversacional"])
TODAS = CARTERA + CONVERSACIONAL
print(f"\ncartera: {len(CARTERA)} | conversacional: {len(CONVERSACIONAL)} | total: {len(TODAS)}")
print("\nNota: activo_ya_pague queda FUERA de los tres bloques por fuga temporal")
print("(mediana 0 dias respecto de fecha_pago). Ver notebook 03.")

,bloque,variable,descripcion,origen
0,cartera,dias_mora_inicial,Días de mora al inicio del mes,BigQuery
1,cartera,meses_consecutivos_mora,Meses calendario consecutivos en mora,BigQuery
2,cartera,mes_campania,Identificador de campaña del mes,BigQuery
3,cartera,tipo_operacion,TC (tarjeta) o CARTERA,BigQuery
4,cartera,bucket,Tramo de mora asignado por el banco (B1–B7),BigQuery
5,conversacional,total_intenciones,Intenciones activadas en el mes,MySQL (chatbot)
6,conversacional,intenciones_distintas,Intenciones únicas en el mes,MySQL (chatbot)
7,conversacional,activo_compromiso,Activó intención de compromiso de pago,MySQL (chatbot)
8,conversacional,dias_compromiso,"Plazo elegido (1, 3 o 5 días)",MySQL (chatbot)
9,conversacional,ultima_intencion_antes_compromiso,Intención previa al primer compromiso,MySQL (chatbot)


tabla -> outputs\tablas\tabla4_composicion_bloques.csv

cartera: 5 | conversacional: 8 | total: 13

Nota: activo_ya_pague queda FUERA de los tres bloques por fuga temporal
(mediana 0 dias respecto de fecha_pago). Ver notebook 03.


---
## 2 · Tabla 1: Ablación por bloques

Cada modelo se **reentrena desde cero** con su subconjunto de variables, sobre el mismo split
temporal, y se evalúa sobre el mismo test. La diferencia entre filas mide el aporte del bloque.

In [4]:
def entrenar_logistica(feats):
    num = [c for c in feats if c not in CATEGORICAS]
    cat = [c for c in feats if c in CATEGORICAS]
    pasos = [("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                               ("sc", StandardScaler())]), num)]
    if cat:
        pasos.append(("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                       ("oh", OneHotEncoder(handle_unknown="ignore",
                                                            min_frequency=20))]), cat))
    modelo = Pipeline([("pre", ColumnTransformer(pasos)),
                       ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                                  random_state=SEMILLA))])
    modelo.fit(tr[feats], y_tr)
    return modelo.predict_proba(te[feats])[:, 1]


def entrenar_xgboost(feats, hiper=None):
    """Sin `hiper` usa los del modelo operativo; el conjunto completo pasa los suyos."""
    hiper = HIPER_OPERATIVO if hiper is None else hiper
    def prep(d):
        X = d[feats].copy()
        for c in feats:
            X[c] = X[c].astype("category") if c in CATEGORICAS else pd.to_numeric(X[c], errors="coerce")
        return X
    modelo = xgb.XGBClassifier(objective="binary:logistic", eval_metric="aucpr",
                               tree_method="hist", enable_categorical=True,
                               scale_pos_weight=SPW, random_state=SEMILLA,
                               n_jobs=4, **hiper).fit(prep(tr), y_tr)
    return modelo.predict_proba(prep(te))[:, 1]


def metricas(y, p):
    return {"PR_AUC": average_precision_score(y, p),
            "ROC_AUC": roc_auc_score(y, p),
            "Brier": brier_score_loss(y, p),
            "lift_PR": average_precision_score(y, p) / TASA_BASE}


conjuntos = {
    "Solo cartera": CARTERA,
    "Solo conversacional": CONVERSACIONAL,
    "Ambos bloques": TODAS,
}

filas = []
for nombre, feats in conjuntos.items():
    for familia, fn in [("Regresión logística", entrenar_logistica),
                        ("XGBoost", entrenar_xgboost)]:
        m = metricas(y_te, fn(feats))
        filas.append({"conjunto": nombre, "n_variables": len(feats),
                      "modelo": familia, **m})

tabla1 = pd.DataFrame(filas)
display(tabla1.round(4))

,conjunto,n_variables,modelo,PR_AUC,ROC_AUC,Brier,lift_PR
0,Solo cartera,5,Regresión logística,0.3191,0.7294,0.1592,1.8499
1,Solo cartera,5,XGBoost,0.3393,0.7353,0.1687,1.9673
2,Solo conversacional,8,Regresión logística,0.3492,0.6834,0.2263,2.0244
3,Solo conversacional,8,XGBoost,0.3418,0.6797,0.2224,1.9814
4,Ambos bloques,13,Regresión logística,0.4209,0.7644,0.1421,2.4400
5,Ambos bloques,13,XGBoost,0.3875,0.7723,0.1634,2.2465


In [5]:
# Version ancha, que es la que se lee mejor en un documento
ancha = tabla1.pivot(index=["conjunto", "n_variables"], columns="modelo",
                     values="PR_AUC").reset_index()
ancha.columns.name = None
orden = ["Solo cartera", "Solo conversacional", "Ambos bloques"]
ancha = ancha.set_index("conjunto").reindex(orden).reset_index()

base_log = ancha.loc[ancha["conjunto"] == "Solo cartera", "Regresión logística"].iloc[0]
base_xgb = ancha.loc[ancha["conjunto"] == "Solo cartera", "XGBoost"].iloc[0]
ancha["aporte_logistica"] = ancha["Regresión logística"] - base_log
ancha["aporte_xgboost"] = ancha["XGBoost"] - base_xgb
ancha["lift_logistica"] = ancha["Regresión logística"] / TASA_BASE
ancha["lift_xgboost"] = ancha["XGBoost"] / TASA_BASE

display(ancha.round(4))
exportar(ancha.round(4), "tabla1_ablacion_bloques")

print(f"\nTasa base en test (PR-AUC del azar): {TASA_BASE:.4f}")
print(f"\nAporte del bloque conversacional sobre solo cartera:")
fila = ancha[ancha["conjunto"] == "Ambos bloques"].iloc[0]
print(f"   Regresion logistica: {base_log:.4f} -> {fila['Regresión logística']:.4f}  "
      f"({fila['aporte_logistica']:+.4f})")
print(f"   XGBoost            : {base_xgb:.4f} -> {fila['XGBoost']:.4f}  "
      f"({fila['aporte_xgboost']:+.4f})")
conv = ancha[ancha["conjunto"] == "Solo conversacional"].iloc[0]
print(f"\nEl bloque conversacional SOLO frente a la cartera SOLA:")
print(f"   Regresion logistica: {conv['Regresión logística']:.4f} vs {base_log:.4f}  "
      f"({conv['Regresión logística'] - base_log:+.4f})")
print(f"   XGBoost            : {conv['XGBoost']:.4f} vs {base_xgb:.4f}  "
      f"({conv['XGBoost'] - base_xgb:+.4f})")

,conjunto,n_variables,Regresión logística,XGBoost,aporte_logistica,aporte_xgboost,lift_logistica,lift_xgboost
0,Solo cartera,5,0.3191,0.3393,0.0000,0.0000,1.8499,1.9673
1,Solo conversacional,8,0.3492,0.3418,0.0301,0.0024,2.0244,1.9814
2,Ambos bloques,13,0.4209,0.3875,0.1018,0.0482,2.4400,2.2465


tabla -> outputs\tablas\tabla1_ablacion_bloques.csv

Tasa base en test (PR-AUC del azar): 0.1725

Aporte del bloque conversacional sobre solo cartera:
   Regresion logistica: 0.3191 -> 0.4209  (+0.1018)
   XGBoost            : 0.3393 -> 0.3875  (+0.0482)

El bloque conversacional SOLO frente a la cartera SOLA:
   Regresion logistica: 0.3492 vs 0.3191  (+0.0301)
   XGBoost            : 0.3418 vs 0.3393  (+0.0024)


**Lectura de la tabla 1.** El bloque conversacional por sí solo alcanza un PR-AUC superior al del
bloque de cartera en la regresión logística, y prácticamente igual en XGBoost. Añadirlo a la cartera
mejora en ambos modelos. Es la evidencia directa de la hipótesis del trabajo: **la señal está en las
variables conversacionales, y el resultado no depende del algoritmo.**

---
## 3 · Tabla 2: Comparativa de métricas por configuración

Ocho configuraciones de **tres familias** (referencia, regresión logística y XGBoost). No son ocho
algoritmos distintos: es una comparativa de configuraciones, no un análisis de alternativas
técnicas.

In [6]:
metricas_guardadas = json.loads((DIR_OUTPUTS / "metricas_modelo_v1.json").read_text(encoding="utf-8"))
test = metricas_guardadas["test"]

FAMILIA = {
    "Tasa base (dummy)": ("DummyClassifier", "—", "referencia"),
    "Logistica operativa": ("LogisticRegression", "13 (operativo)", "—"),
    "Logistica completa": ("LogisticRegression", "14 (completo)", "—"),
    "XGBoost operativo · CV aleatoria": ("XGBClassifier", "13 (operativo)", "StratifiedGroupKFold"),
    "XGBoost operativo · CV temporal": ("XGBClassifier", "13 (operativo)", "ventana expansiva"),
    "XGBoost operativo · monotono": ("XGBClassifier", "13 (operativo)", "temporal + monotonía"),
    "XGBoost completo · CV temporal": ("XGBClassifier", "14 (completo)", "ventana expansiva"),
    "XGBoost completo · monotono": ("XGBClassifier", "14 (completo)", "temporal + monotonía"),
}

filas = []
for nombre, m in test.items():
    familia, variables, seleccion = FAMILIA.get(nombre, ("—", "—", "—"))
    filas.append({
        "modelo": nombre, "familia": familia, "variables": variables,
        "seleccion_hiperparametros": seleccion,
        "PR_AUC": m["PR_AUC"], "ROC_AUC": m["ROC_AUC"], "Brier": m["Brier"],
        "lift_PR": m["PR_AUC"] / TASA_BASE,
    })

tabla2 = (pd.DataFrame(filas)
          .sort_values("PR_AUC", ascending=False)
          .reset_index(drop=True))
tabla2.insert(0, "puesto", range(1, len(tabla2) + 1))
display(tabla2.round(4))
exportar(tabla2.round(4), "tabla2_comparativa_modelos")

print(f"\nTasa base en test: {TASA_BASE:.4f}, que es el PR-AUC de un clasificador aleatorio.")
print("Brier mas bajo = mejor calibrado. PR-AUC mas alto = mejor ordenamiento.")

,puesto,modelo,familia,variables,seleccion_hiperparametros,PR_AUC,ROC_AUC,Brier,lift_PR
0,1,Logistica completa,LogisticRegression,14 (completo),—,0.4543,0.7724,0.1377,2.6340
1,2,XGBoost completo · CV temporal,XGBClassifier,14 (completo),ventana expansiva,0.4458,0.7912,0.1616,2.5845
2,3,Logistica operativa,LogisticRegression,13 (operativo),—,0.4209,0.7644,0.1421,2.4400
3,4,XGBoost completo · monotono,XGBClassifier,14 (completo),temporal + monotonía,0.4067,0.7816,0.1623,2.3578
4,5,XGBoost operativo · CV temporal,XGBClassifier,13 (operativo),ventana expansiva,0.3923,0.7751,0.1635,2.2742
5,6,XGBoost operativo · monotono,XGBClassifier,13 (operativo),temporal + monotonía,0.3895,0.7737,0.1665,2.2579
6,7,XGBoost operativo · CV aleatoria,XGBClassifier,13 (operativo),StratifiedGroupKFold,0.3782,0.7648,0.1601,2.1926
7,8,Tasa base (dummy),DummyClassifier,—,referencia,0.1725,0.5000,0.1457,1.0000


tabla -> outputs\tablas\tabla2_comparativa_modelos.csv

Tasa base en test: 0.1725, que es el PR-AUC de un clasificador aleatorio.
Brier mas bajo = mejor calibrado. PR-AUC mas alto = mejor ordenamiento.


---
## 4 · Tabla 3: Significancia estadística

Una diferencia de PR-AUC no significa nada sin saber si sobrevive al error de muestreo. Bootstrap
pareado: se remuestrea el test con reemplazo y se evalúan ambos modelos sobre **las mismas filas**
en cada réplica.

In [7]:
FEATURES_OPERATIVO = TODAS
FEATURES_COMPLETO = TODAS + ["activo_ya_pague"]

pred = {
    "Logística operativa": entrenar_logistica(FEATURES_OPERATIVO),
    "XGBoost operativo": entrenar_xgboost(FEATURES_OPERATIVO, HIPER_OPERATIVO),
    "Logística completa": entrenar_logistica(FEATURES_COMPLETO),
    "XGBoost completo": entrenar_xgboost(FEATURES_COMPLETO, HIPER_COMPLETO),
}
for k, v in pred.items():
    print(f"{k:<22} PR-AUC {average_precision_score(y_te, v):.4f}")

rng = np.random.default_rng(SEMILLA)
B, n = 2000, len(y_te)

def bootstrap(a, b):
    difs = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        if y_te[idx].sum() < 10:
            continue
        difs.append(average_precision_score(y_te[idx], pred[a][idx]) -
                    average_precision_score(y_te[idx], pred[b][idx]))
    difs = np.array(difs)
    lo, hi = np.percentile(difs, [2.5, 97.5])
    return {"modelo_A": a, "modelo_B": b,
            "PR_AUC_A": average_precision_score(y_te, pred[a]),
            "PR_AUC_B": average_precision_score(y_te, pred[b]),
            "diferencia_media": difs.mean(), "IC95_inferior": lo, "IC95_superior": hi,
            "P_gana_A": (difs > 0).mean(),
            "significativa": "Sí" if (lo > 0 or hi < 0) else "No (el IC cruza cero)"}

tabla3 = pd.DataFrame([
    bootstrap("Logística operativa", "XGBoost operativo"),
    bootstrap("Logística completa", "XGBoost completo"),
])
display(tabla3.round(4))
exportar(tabla3.round(4), "tabla3_significancia")

print("\nLectura: la ventaja de la linea base es significativa en el conjunto OPERATIVO")
print("que es el unico desplegable, y no lo es en el COMPLETO, que incluye la variable con")
print("fuga temporal. En el completo los dos modelos estan empatados estadisticamente.")

Logística operativa    PR-AUC 0.4209
XGBoost operativo      PR-AUC 0.3875
Logística completa     PR-AUC 0.4543
XGBoost completo       PR-AUC 0.4451


,modelo_A,modelo_B,PR_AUC_A,PR_AUC_B,diferencia_media,IC95_inferior,IC95_superior,P_gana_A,significativa
0,Logística operativa,XGBoost operativo,0.4209,0.3875,0.0332,0.0182,0.0483,1.0000,Sí
1,Logística completa,XGBoost completo,0.4543,0.4451,0.0094,-0.0035,0.0232,0.9145,No (el IC cruza cero)


tabla -> outputs\tablas\tabla3_significancia.csv

Lectura: la ventaja de la linea base es significativa en el conjunto OPERATIVO
que es el unico desplegable, y no lo es en el COMPLETO, que incluye la variable con
fuga temporal. En el completo los dos modelos estan empatados estadisticamente.


---
## 5 · Índice de lo exportado

In [8]:
print("=" * 72)
print("TABLAS EN outputs/tablas/")
print("=" * 72)
for ruta in sorted(DIR_TABLAS.glob("*.csv")):
    t = pd.read_csv(ruta)
    print(f"  {ruta.name:<38} {t.shape[0]:>3} filas x {t.shape[1]} columnas")

print()
print("=" * 72)
print("FIGURAS EN outputs/figuras/")
print("=" * 72)
for ruta in sorted((DIR_OUTPUTS / "figuras").glob("*.png")):
    print(f"  {ruta.name}")

TABLAS EN outputs/tablas/
  tabla1_ablacion_bloques.csv              3 filas x 8 columnas
  tabla2_comparativa_modelos.csv           8 filas x 9 columnas
  tabla3_significancia.csv                 2 filas x 9 columnas
  tabla4_composicion_bloques.csv          13 filas x 4 columnas

FIGURAS EN outputs/figuras/
  00_distribucion_target.png
  01_tasa_cumplimiento_por_mes.png
  02_cobertura_chatbot_por_mes.png
  03_tasa_por_bucket.png
  04_tasa_por_tipo_operacion.png
  05_tasa_por_rango_mora.png
  06_tasa_por_racha_mora.png
  07_tasa_por_contacto_chatbot.png
  08_tasa_por_compromiso.png
  09_tasa_por_ya_pague.png
  10_tasa_por_plazo_compromiso.png
  11_tasa_por_volumen_intenciones.png
  12_tasa_por_hora.png
  13_tasa_por_dia_semana.png
  14_tasa_por_intencion_previa.png
  15_matriz_correlacion.png
  16_informacion_mutua.png
  17_brecha_controlada_por_bucket.png
  18_curvas_pr_roc.png
  19_curva_lift.png
  20_calibracion.png
  21_importancia_ganancia.png
  22_shap_importancia.png
  23_shap_

---
### Cómo usar estas tablas

Los CSV están en `outputs/tablas/` con codificación `utf-8-sig`, así que **Excel los abre con los
acentos correctos** al hacer doble clic. Desde ahí se copian a Word manteniendo el formato de tabla.

Las cifras se recalculan cada vez que se ejecuta este notebook: si algo cambia aguas arriba
(el dataset, el modelo o los hiperparámetros) las tablas del documento se regeneran desde la misma
fuente y no quedan desincronizadas.